**STATUS:** ALIVE (descriptive)
**LAST-AUDIT:** 2026-05-05
**FEEDS:** B12, B13, B14 (the post-pivot headline bid-shape evidence); B8 (cross-validation with the IB-canonical study in nb09)
**CLAIM:** Comprehensive descriptive atlas of OMIE bid behaviour across regimes, technologies, firms, plants, hours of day, and seasons. Anchors the within-market granularity mechanism (`notebooks/memos/_within_market_granularity_model.md`) at the descriptive level.


# Bid-shape atlas — how do bids look across every dimension?

Workshop feedback (May 5 CEMFI): bid-shape is the most important piece of the project. The within-market granularity mechanism predicts that operators bid differently across the four quarters of an hour in critical (steep within-hour profile) but not in flat (constant within-hour profile) hours, and that this should be visible in the SHAPE of bids: more tranches, steeper ladder, lower average price in critical hours.

This notebook surveys bid behaviour across every relevant dimension:

1. **Across regimes** — pre-DA15-reform → 3-sess → ISP15-win → DA60/ID15 (pre/post blackout) → DA15/ID15.
2. **By technology** — CCGT, Hydro, Nuclear, Wind, Solar.
3. **By firm** — dominant (Iberdrola, Endesa, Naturgy, EDP) vs fringe.
4. **By plant** — drilldown on top units in each technology.
5. **By hour of day** — critical hours h{7,8,16,17,18} vs flat hours h{3,4,5}.
6. **By season** — month-of-year × regime patterns.
7. **Within-hour granularity exploitation** — post-MTU15-DA, do operators actually use the 4-quarter granularity, or do they mechanically repeat the same hourly bid across quarters?

Two markets are covered:

- **IDA bids** (IDET joined to ICAB) — the primary panel because it spans the full reform sequence cleanly. DA bid-prices pre-2025-03-19 are 0-padded due to a parser artefact, so DA bids are restricted to post-MTU15-IDA observations.
- **DA bids** (DET joined to CAB), post-MTU15-IDA — used specifically for the within-hour granularity exploitation analysis (which requires the four-quarter post-MTU15-DA panel).

Bid-shape measures throughout:

- **Number of distinct price tranches** per offer (offer complexity).
- **Price range** = $p_{\max} - p_{\min}$ per offer (high-end aggressiveness span).
- **Ladder slope** = $(p_{\max} - p_{\min}) / Q_{\text{total}}$ (price-per-MW steepness).
- **Quantity-weighted average price** $\bar{p} = \sum_t p_t q_t / \sum_t q_t$.
- **Total quantity offered** $Q_{\text{total}} = \sum_t q_t$.


In [ ]:
# Setup
from __future__ import annotations
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path('/Users/pabloparamio/Desktop/CEMFI/2nd Year/Master Thesis/mtu15_project')
DET   = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_diario' / 'ofertas' / 'det_all.parquet'
CAB   = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_diario' / 'ofertas' / 'cab_all.parquet'
IDET  = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_intradiario_subastas' / 'ofertas' / 'idet_all.parquet'
ICAB  = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_intradiario_subastas' / 'ofertas' / 'icab_all.parquet'
PDBCE = PROJECT / 'data' / 'processed' / 'omie' / 'mercado_diario' / 'programas' / 'pdbce_all.parquet'
LISTA = PROJECT / 'data' / 'external' / 'omie_reference' / 'lista_unidades.csv'
OUT   = PROJECT / 'figures' / 'working'
OUT.mkdir(exist_ok=True, parents=True)

# Reform regime cuts (one constant block, used everywhere)
PRE_IDA_END    = pd.Timestamp('2024-06-14')
THREE_SESS_END = pd.Timestamp('2024-12-01')
ISP15_END      = pd.Timestamp('2025-03-19')
BLACKOUT       = pd.Timestamp('2025-04-28')
MTU15_DA_DATE  = pd.Timestamp('2025-10-01')

CRITICAL_HOURS = [7, 8, 16, 17, 18]
FLAT_HOURS     = [3, 4, 5]
DOMINANT       = ['IB', 'GE', 'GN', 'HC']  # HC = EDP España legacy code

REGIMES_6 = ['pre-DA15', '3-sess', 'ISP15-win', 'DA60/ID15-prebo', 'DA60/ID15-reforz', 'DA15/ID15']

def regime_of(d):
    d = pd.Timestamp(d)
    if d < PRE_IDA_END:    return 'pre-DA15'
    if d < THREE_SESS_END: return '3-sess'
    if d < ISP15_END:      return 'ISP15-win'
    if d < BLACKOUT:       return 'DA60/ID15-prebo'
    if d < MTU15_DA_DATE:  return 'DA60/ID15-reforz'
    return 'DA15/ID15'

con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=4")
print('OK — paths and constants set.')


In [ ]:
# Firm + tech mapping per unit_code

# 1. firm from PDBCE grupo_empresarial (thermal/hydro/nuclear): latest per unit
firms_thermal = con.execute(f'''
    SELECT unit_code, grupo_empresarial AS firm FROM (
      SELECT unit_code, grupo_empresarial,
             ROW_NUMBER() OVER (PARTITION BY unit_code ORDER BY date DESC) AS rn
      FROM '{PDBCE}' WHERE grupo_empresarial IS NOT NULL) WHERE rn = 1
''').df()

# 2. tech + owner_agent from LISTA_UNIDADES
units = pd.read_csv(LISTA)[['unit_code', 'technology', 'owner_agent']]

def tech_bucket(t):
    if t is None or pd.isna(t): return 'unknown'
    t = str(t)
    if 'Solar' in t: return 'solar'
    if 'Eólica' in t: return 'wind'
    if 'Hidráulica' in t or 'Hidraulic' in t: return 'hydro'
    if 'Ciclo Combinado' in t: return 'CCGT'
    if 'Nuclear' in t: return 'nuclear'
    if 'Térmica no Renovab' in t: return 'thermal_nonRE'
    if 'Térmica Renovable' in t: return 'thermal_RE'
    if 'Almacen' in t: return 'storage'
    if 'Comerc' in t or 'Compra' in t or 'Consumi' in t or 'Import' in t: return 'comm/buyer'
    return 'other'

def owner_to_firm(s):
    if s is None or pd.isna(s): return None
    s = str(s).upper()
    if 'IBERDROLA' in s: return 'IB'
    if 'ENDESA' in s or 'ENEL GREEN POWER' in s: return 'GE'
    if 'NATURGY' in s or 'GAS NATURAL' in s: return 'GN'
    if 'EDP' in s or 'HIDROELÉCTRICA DEL CANTÁBRICO' in s or 'HC ENERG' in s: return 'HC'
    return None

units['tech'] = units['technology'].apply(tech_bucket)
units['firm_from_owner'] = units['owner_agent'].apply(owner_to_firm)

# 3. combine: prefer grupo_empresarial, fall back to owner_agent for renewables
m = units.merge(firms_thermal, on='unit_code', how='left')
m['firm'] = m['firm'].fillna(m['firm_from_owner'])
m['firm_class'] = np.where(m['firm'].isin(DOMINANT), 'dominant', 'fringe')

unit_map = m[['unit_code', 'firm', 'tech', 'firm_class']].dropna(subset=['firm'])
con.register('unit_map', unit_map)

print(f'unit_map rows: {len(unit_map):,}')
print(unit_map.groupby(['firm_class','tech']).size().unstack(fill_value=0))


## Section 1 — Bid panel construction

For each (date, period, unit) we compute five bid-shape measures from the offer-detail tables (DET / IDET) joined to the offer headers (CAB / ICAB) for unit-code lookup:

- `n_tranches` — distinct price points
- `q_total` — total quantity offered
- `p_min`, `p_max`, `p_avg` — min, max, qty-weighted mean price
- `slope = (p_max - p_min) / q_total` — ladder steepness

We use IDA bids as the primary panel because they cleanly span all six regimes; DA bid prices pre-2025-03-19 are 0-padded due to a parser artefact and are excluded from the cross-regime view. DA bids are used in §9 for the within-hour granularity exploitation analysis.


In [ ]:
# IDA bid panel — sell offers (buy_sell='V') across all regimes
# IDET has unit_code directly; ICAB is used only to filter buy_sell='V' on the offer header

ida_bids = con.execute(f"""
    WITH icab_latest AS (
      SELECT CAST(date AS DATE) AS d, session_number, offer_code, version,
             ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), session_number, offer_code
                                ORDER BY version DESC) AS rn
      FROM '{ICAB}' WHERE buy_sell = 'V'
    ),
    icab_v AS (SELECT * FROM icab_latest WHERE rn = 1),
    idet_clean AS (
      SELECT CAST(date AS DATE) AS d, session_number, offer_code, version, period, unit_code,
             price_eur_mwh AS price, quantity_mw AS qty
      FROM '{IDET}'
      WHERE quantity_mw IS NOT NULL AND quantity_mw > 0
        AND price_eur_mwh IS NOT NULL
        AND period BETWEEN 1 AND 96
    )
    SELECT idet.d AS date, idet.session_number, idet.period, idet.unit_code,
           unit_map.firm, unit_map.tech, unit_map.firm_class,
           COUNT(DISTINCT idet.price) AS n_tranches,
           SUM(idet.qty)              AS q_total,
           MIN(idet.price)            AS p_min,
           MAX(idet.price)            AS p_max,
           SUM(idet.price * idet.qty) / SUM(idet.qty) AS p_avg
    FROM idet_clean idet
      JOIN icab_v USING (d, session_number, offer_code, version)
      JOIN unit_map ON idet.unit_code = unit_map.unit_code
    GROUP BY 1, 2, 3, 4, 5, 6, 7
""").df()

ida_bids['date'] = pd.to_datetime(ida_bids['date'])
ida_bids['regime'] = ida_bids['date'].apply(regime_of)
ida_bids['slope']  = (ida_bids['p_max'] - ida_bids['p_min']) / ida_bids['q_total'].clip(lower=1.0)
ida_bids['p_range'] = ida_bids['p_max'] - ida_bids['p_min']

# Hour-of-day: IDA goes 15-min at MTU15-IDA (2025-03-19); pre that, periods are hourly
ida_bids['hour'] = np.where(
    ida_bids['date'] >= ISP15_END,
    (ida_bids['period'] - 1) // 4,
    (ida_bids['period'] - 1)
).clip(0, 23)

print(f'IDA bid-panel rows: {len(ida_bids):,}')
print(f'Distinct units: {ida_bids.unit_code.nunique():,}')
print(ida_bids.groupby('regime').size().reindex(REGIMES_6))


In [ ]:
# DA bid panel — sell offers post-MTU15-IDA (when DET prices are no longer 0-padded)

da_bids = con.execute(f'''
    WITH cab_latest AS (
      SELECT CAST(date AS DATE) AS d, offer_code, version, unit_code,
             ROW_NUMBER() OVER (PARTITION BY CAST(date AS DATE), offer_code, unit_code
                                ORDER BY version DESC) AS rn
      FROM '{CAB}'
      WHERE buy_sell = 'V'
        AND CAST(date AS DATE) >= DATE '2025-03-19'
    ),
    cab_unique AS (SELECT * FROM cab_latest WHERE rn = 1),
    det_clean AS (
      SELECT CAST(date AS DATE) AS d, offer_code, version, period,
             price_eur_mwh AS price, quantity_mw AS qty
      FROM '{DET}'
      WHERE quantity_mw IS NOT NULL AND quantity_mw > 0
        AND price_eur_mwh IS NOT NULL
        AND CAST(date AS DATE) >= DATE '2025-03-19'
        AND period BETWEEN 1 AND 96
    )
    SELECT det.d AS date, det.period, cab_unique.unit_code,
           unit_map.firm, unit_map.tech, unit_map.firm_class,
           COUNT(DISTINCT det.price) AS n_tranches,
           SUM(det.qty)              AS q_total,
           MIN(det.price)            AS p_min,
           MAX(det.price)            AS p_max,
           SUM(det.price * det.qty) / SUM(det.qty) AS p_avg
    FROM det_clean det
      JOIN cab_unique USING (d, offer_code, version)
      JOIN unit_map ON cab_unique.unit_code = unit_map.unit_code
    GROUP BY 1, 2, 3, 4, 5, 6
''').df()

da_bids['date']    = pd.to_datetime(da_bids['date'])
da_bids['regime']  = da_bids['date'].apply(regime_of)
da_bids['slope']   = (da_bids['p_max'] - da_bids['p_min']) / da_bids['q_total'].clip(lower=1.0)
da_bids['p_range'] = da_bids['p_max'] - da_bids['p_min']

# DA goes 15-min only at MTU15-DA (2025-10-01); hour mapping accordingly
da_bids['hour'] = np.where(
    da_bids['date'] >= MTU15_DA_DATE,
    (da_bids['period'] - 1) // 4,
    (da_bids['period'] - 1)
).clip(0, 23)
da_bids['quarter'] = np.where(
    da_bids['date'] >= MTU15_DA_DATE,
    (da_bids['period'] - 1) % 4,
    0
)

print(f'DA bid-panel rows (post-2025-03-19): {len(da_bids):,}')
print(f'Distinct units: {da_bids.unit_code.nunique():,}')
print(da_bids.groupby('regime').size().reindex(REGIMES_6))


## Section 2 — Bid measures across regimes (overall)

How does the average bid look across the six reform regimes? We aggregate the IDA panel by regime and report the median and P90 of each measure (medians are insensitive to extreme tranches; P90 captures the strategic tail).


In [ ]:
# Aggregate IDA bids by regime
def regime_agg(df, group_cols):
    g = df.groupby(group_cols)
    out = pd.DataFrame({
        'n_offers':   g.size(),
        'tranches_med':  g['n_tranches'].median(),
        'tranches_p90':  g['n_tranches'].quantile(0.90),
        'q_total_med':   g['q_total'].median(),
        'p_avg_med':     g['p_avg'].median(),
        'p_range_med':   g['p_range'].median(),
        'p_range_p90':   g['p_range'].quantile(0.90),
        'slope_med':     g['slope'].median(),
        'slope_p90':     g['slope'].quantile(0.90),
    }).reset_index()
    return out

reg_summary = regime_agg(ida_bids, ['regime'])
reg_summary = reg_summary.set_index('regime').reindex(REGIMES_6).reset_index()
print('IDA bid measures by regime (medians + P90 of strategic tail):')
print(reg_summary.round(2).to_string(index=False))

# Plot: 4 panels — tranches, p_range (P90), slope (P90), q_total
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (col, title) in zip(axes, [
    ('tranches_med', 'tranches per offer (median)'),
    ('p_range_p90',  'price range €/MWh (P90)'),
    ('slope_p90',    'ladder slope €/MW per MWh (P90)'),
    ('q_total_med',  'q_total MWh (median)'),
]):
    ax.bar(reg_summary['regime'], reg_summary[col], color='steelblue')
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.3)
plt.suptitle('IDA bid measures across regimes (all units, all tech)', fontsize=11)
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_01_overall_regime.png', dpi=110, bbox_inches='tight')
plt.show()


## Section 3 — Bid measures by technology × regime

For each (tech, regime) pair, what does the average bid look like? Particularly: do CCGT and Hydro (the dispatchable techs that the workshop bid-shape result was concentrated in) show distinct patterns from Wind, Solar, and Nuclear?


In [ ]:
# Per-tech × regime aggregation
TECHS_PRIMARY = ['CCGT', 'hydro', 'nuclear', 'wind', 'solar']

tech_reg = regime_agg(ida_bids[ida_bids['tech'].isin(TECHS_PRIMARY)], ['tech', 'regime'])

# Pivot for plotting
def grid_plot(measure, title, log=False):
    fig, axes = plt.subplots(1, 5, figsize=(16, 3.4), sharey=True)
    for ax, t in zip(axes, TECHS_PRIMARY):
        sub = tech_reg[tech_reg['tech'] == t].set_index('regime').reindex(REGIMES_6).reset_index()
        ax.bar(sub['regime'], sub[measure], color='steelblue')
        ax.set_title(t, fontsize=10)
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        ax.grid(alpha=0.3)
        if log: ax.set_yscale('log')
    plt.suptitle(title, fontsize=11)
    plt.tight_layout()

grid_plot('tranches_med', 'Tranches per offer (median) — IDA bids by tech × regime')
plt.savefig(OUT / 'bid_atlas_02_tech_tranches.png', dpi=110, bbox_inches='tight')
plt.show()

grid_plot('p_range_p90', 'Price range €/MWh (P90) — IDA bids by tech × regime', log=True)
plt.savefig(OUT / 'bid_atlas_03_tech_prange.png', dpi=110, bbox_inches='tight')
plt.show()

grid_plot('p_avg_med', 'Avg price €/MWh (median) — IDA bids by tech × regime')
plt.savefig(OUT / 'bid_atlas_04_tech_pavg.png', dpi=110, bbox_inches='tight')
plt.show()


## Section 4 — Bid measures by firm × regime

Two questions:

1. **Dominant vs Fringe**: do dominant firms (IB, GE, GN, EDP/HC) bid systematically differently from the fringe? This is the cross-validation of the workshop's fringe-placebo result at the bid-shape level.
2. **Per-dominant-firm**: are the four dominant firms structurally different from each other?


In [ ]:
# Dominant vs Fringe
dom_reg = regime_agg(ida_bids, ['firm_class', 'regime'])
print('IDA bid measures by firm class × regime:')
print(dom_reg.round(2).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (col, title) in zip(axes, [
    ('tranches_med', 'tranches per offer (median)'),
    ('p_range_p90',  'price range €/MWh (P90)'),
    ('slope_p90',    'ladder slope (P90)'),
]):
    pivot = dom_reg.pivot(index='regime', columns='firm_class', values=col).reindex(REGIMES_6)
    pivot.plot(kind='bar', ax=ax, color=['tab:blue', 'tab:orange'])
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.3)
    ax.set_xlabel('')
plt.suptitle('IDA bid measures: Dominant vs Fringe across regimes', fontsize=11)
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_05_dom_vs_fringe.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# Per dominant firm × regime
dom_only = ida_bids[ida_bids['firm_class'] == 'dominant']
firm_reg = regime_agg(dom_only, ['firm', 'regime'])

fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
for ax, f in zip(axes, DOMINANT):
    sub = firm_reg[firm_reg['firm'] == f].set_index('regime').reindex(REGIMES_6).reset_index()
    ax.bar(sub['regime'], sub['tranches_med'], color='steelblue')
    ax.set_title(f, fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.grid(alpha=0.3)
plt.suptitle('Tranches per offer (median) — IDA bids by dominant firm × regime', fontsize=11)
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_06_dominant_per_firm.png', dpi=110, bbox_inches='tight')
plt.show()


## Section 5 — Hour-of-day patterns

Heatmap of bid measures by hour of day × regime. Critical hours h{7,8,16,17,18} should show distinct patterns from flat hours h{3,4,5} once the granularity reform takes hold.


In [ ]:
# Hour-of-day × regime heatmaps (IDA bids, all units pooled)
hour_reg = (ida_bids.groupby(['hour', 'regime'])
                    .agg(tranches=('n_tranches', 'median'),
                         prange=('p_range', lambda s: s.quantile(0.90)),
                         slope=('slope', lambda s: s.quantile(0.90)),
                         pavg=('p_avg', 'median'))
                    .reset_index())

def heatmap(df, val, title, cmap='viridis'):
    pivot = df.pivot(index='hour', columns='regime', values=val).reindex(columns=REGIMES_6)
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap, origin='lower')
    ax.set_yticks(range(24)); ax.set_yticklabels(range(24), fontsize=8)
    ax.set_xticks(range(len(REGIMES_6))); ax.set_xticklabels(REGIMES_6, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Hour of day')
    # mark critical hours
    for h in CRITICAL_HOURS:
        ax.axhline(h, color='red', alpha=0.25, lw=0.6)
    for h in FLAT_HOURS:
        ax.axhline(h, color='blue', alpha=0.25, lw=0.6)
    plt.colorbar(im, ax=ax, label=val)
    ax.set_title(f'{title}\nred=critical hours, blue=flat hours', fontsize=10)
    plt.tight_layout()

heatmap(hour_reg, 'tranches', 'IDA bid tranches per offer (median)')
plt.savefig(OUT / 'bid_atlas_07_hour_tranches.png', dpi=110, bbox_inches='tight')
plt.show()

heatmap(hour_reg, 'prange', 'IDA bid price range €/MWh (P90)', cmap='magma')
plt.savefig(OUT / 'bid_atlas_08_hour_prange.png', dpi=110, bbox_inches='tight')
plt.show()


## Section 6 — Critical-flat differential (the headline mechanism)

For each (tech, regime) we compute the critical-flat differential in the bid measures. Under the within-market granularity mechanism, the differential should be positive and growing where granularity has economic content (CCGT, hydro) and ~zero where it doesn't (wind, solar, nuclear).


In [ ]:
# Critical-flat differential by tech × regime
def crit_flat_diff(df, group_cols, measure, fn='median'):
    df = df.copy()
    df['hour_class'] = np.where(df['hour'].isin(CRITICAL_HOURS), 'critical',
                          np.where(df['hour'].isin(FLAT_HOURS), 'flat', 'other'))
    df = df[df['hour_class'].isin(['critical', 'flat'])]
    if fn == 'median':
        agg = df.groupby(group_cols + ['hour_class'])[measure].median().unstack('hour_class')
    elif fn == 'p90':
        agg = df.groupby(group_cols + ['hour_class'])[measure].quantile(0.90).unstack('hour_class')
    agg['diff'] = agg['critical'] - agg['flat']
    return agg.reset_index()

cf_tech = crit_flat_diff(ida_bids[ida_bids['tech'].isin(TECHS_PRIMARY)],
                         ['tech', 'regime'], 'n_tranches', 'median')

fig, ax = plt.subplots(figsize=(11, 4))
pivot = cf_tech.pivot(index='regime', columns='tech', values='diff').reindex(REGIMES_6)
pivot.plot(kind='bar', ax=ax, width=0.85)
ax.axhline(0, color='black', lw=0.6)
ax.set_title('Critical-hours − flat-hours differential in IDA bid tranches per offer (median), by tech × regime', fontsize=10)
ax.set_ylabel('crit − flat (median tranches per offer)')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='tech', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_09_crit_flat_tech.png', dpi=110, bbox_inches='tight')
plt.show()

# Same for price range
cf_tech_pr = crit_flat_diff(ida_bids[ida_bids['tech'].isin(TECHS_PRIMARY)],
                             ['tech', 'regime'], 'p_range', 'p90')
fig, ax = plt.subplots(figsize=(11, 4))
pivot = cf_tech_pr.pivot(index='regime', columns='tech', values='diff').reindex(REGIMES_6)
pivot.plot(kind='bar', ax=ax, width=0.85)
ax.axhline(0, color='black', lw=0.6)
ax.set_title('Critical − flat differential in IDA price range €/MWh (P90), by tech × regime', fontsize=10)
ax.set_ylabel('crit − flat (€/MWh, P90)')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='tech', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_10_crit_flat_prange.png', dpi=110, bbox_inches='tight')
plt.show()


## Section 7 — Seasonality (monthly patterns)

Spanish electricity is highly seasonal — solar peaks May–August, wind is winter-heavy, demand peaks in winter and summer. Bid measures should reflect this. We aggregate by month-of-year × regime and report on dispatchable techs (CCGT + hydro), where the within-market granularity mechanism predicts a response.


In [ ]:
# Monthly patterns by tech (focus: CCGT + hydro)
ida_bids['month'] = ida_bids['date'].dt.month

# Per-month × regime for CCGT
def monthly_panel(tech_filter, measure, fn='median'):
    df = ida_bids[ida_bids['tech'].isin(tech_filter)]
    if fn == 'median':
        return df.groupby(['regime', 'month'])[measure].median().unstack('month').reindex(REGIMES_6)
    elif fn == 'p90':
        return df.groupby(['regime', 'month'])[measure].quantile(0.90).unstack('month').reindex(REGIMES_6)

panel_ccgt_tr = monthly_panel(['CCGT'], 'n_tranches', 'median')
panel_hyd_tr  = monthly_panel(['hydro'], 'n_tranches', 'median')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (panel, title) in zip(axes, [(panel_ccgt_tr, 'CCGT'), (panel_hyd_tr, 'Hydro')]):
    im = ax.imshow(panel.values, aspect='auto', cmap='viridis')
    ax.set_yticks(range(len(panel.index))); ax.set_yticklabels(panel.index, fontsize=9)
    ax.set_xticks(range(12)); ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
    ax.set_xlabel('Month')
    ax.set_title(f'{title} tranches per offer (median), by regime × month', fontsize=10)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_11_seasonality_tranches.png', dpi=110, bbox_inches='tight')
plt.show()


## Section 8 — Plant-level drilldown (top units by volume)

For each tech in {CCGT, hydro, nuclear}, identify the top units by total IDA volume and report bid measures across regimes. Reveals plant-level heterogeneity and identifies the units driving aggregate patterns.


In [ ]:
# Top units per tech by total IDA volume
top_units_per_tech = {}
for t in ['CCGT', 'hydro', 'nuclear']:
    top = (ida_bids[ida_bids['tech'] == t]
           .groupby(['unit_code', 'firm'])['q_total'].sum()
           .reset_index().nlargest(8, 'q_total'))
    top_units_per_tech[t] = top
    print(f'\nTop 8 {t} units by total IDA volume:')
    print(top.to_string(index=False))


In [ ]:
# Per-unit bid-shape across regimes for top CCGT units
def per_unit_panel(tech, top_n=6, measure='n_tranches'):
    top = top_units_per_tech[tech].head(top_n)
    sub = ida_bids[ida_bids['unit_code'].isin(top['unit_code'])]
    panel = (sub.groupby(['unit_code', 'regime'])[measure]
                .median().unstack('regime').reindex(columns=REGIMES_6))
    panel = panel.reindex(top['unit_code'].values)  # order by volume
    return panel, top

panel, top = per_unit_panel('CCGT', top_n=8, measure='n_tranches')
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(panel.values, aspect='auto', cmap='viridis')
ax.set_yticks(range(len(panel.index))); ax.set_yticklabels(
    [f'{u} ({top[top.unit_code==u].firm.iloc[0]})' for u in panel.index], fontsize=9)
ax.set_xticks(range(len(REGIMES_6))); ax.set_xticklabels(REGIMES_6, rotation=30, ha='right', fontsize=9)
ax.set_title('Top-8 CCGT units: tranches per offer (median) across regimes', fontsize=10)
plt.colorbar(im, ax=ax, label='median tranches')
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_12_top_ccgt.png', dpi=110, bbox_inches='tight')
plt.show()

panel_h, top_h = per_unit_panel('hydro', top_n=8, measure='n_tranches')
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(panel_h.values, aspect='auto', cmap='viridis')
ax.set_yticks(range(len(panel_h.index))); ax.set_yticklabels(
    [f'{u} ({top_h[top_h.unit_code==u].firm.iloc[0]})' for u in panel_h.index], fontsize=9)
ax.set_xticks(range(len(REGIMES_6))); ax.set_xticklabels(REGIMES_6, rotation=30, ha='right', fontsize=9)
ax.set_title('Top-8 hydro units: tranches per offer (median) across regimes', fontsize=10)
plt.colorbar(im, ax=ax, label='median tranches')
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_13_top_hydro.png', dpi=110, bbox_inches='tight')
plt.show()


## Section 9 — Within-hour granularity exploitation (DA bids post-MTU15-DA)

Post-MTU15-DA (October 2025+), DA bids are submitted per quarter — four bids per hour per unit. The within-market granularity mechanism predicts that operators *should* differentiate across these four quarters in critical hours (where residual production varies within the hour) but may mechanically repeat the same hourly bid in flat hours.

We compute, per (date, hour, unit), whether the four quarters' bids are identical (mechanical repeat) or distinct (granularity exploited). This indicator is the cleanest descriptive evidence for the workshop's bid-shape result.


In [ ]:
# Within-hour granularity: per (date, hour, unit) post-MTU15-DA, is the bid the same across all 4 quarters?
da_post = da_bids[da_bids['date'] >= MTU15_DA_DATE].copy()
print(f'DA bids post-MTU15-DA: {len(da_post):,} (date, period, unit) rows')
print(f'Unique (date, hour, unit): {da_post.groupby(["date","hour","unit_code"]).ngroups:,}')

# For each (date, hour, unit), check if all four quarters have identical n_tranches, p_range, p_avg
g = da_post.groupby(['date', 'hour', 'unit_code'])
hour_unit = pd.DataFrame({
    'firm':       g['firm'].first(),
    'tech':       g['tech'].first(),
    'firm_class': g['firm_class'].first(),
    'n_quarters': g.size(),
    'tranches_var': g['n_tranches'].nunique(),
    'pavg_var':     g['p_avg'].nunique(),
    'prange_var':   g['p_range'].nunique(),
}).reset_index()
hour_unit['hour_class'] = np.where(hour_unit['hour'].isin(CRITICAL_HOURS), 'critical',
                            np.where(hour_unit['hour'].isin(FLAT_HOURS), 'flat', 'other'))
hour_unit['mechanical_repeat'] = ((hour_unit['tranches_var'] == 1) &
                                  (hour_unit['pavg_var'] == 1) &
                                  (hour_unit['prange_var'] == 1)).astype(int)
hour_unit_4q = hour_unit[hour_unit['n_quarters'] == 4]

# Mechanical-repeat rate by tech × hour-class
rate = (hour_unit_4q.groupby(['tech', 'hour_class'])['mechanical_repeat']
        .mean().unstack('hour_class')[['critical', 'flat', 'other']]) * 100
print('\nMechanical-repeat rate (% of unit-hours where all 4 quarters have identical bids), by tech × hour class:')
print(rate.round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
plot_techs = [t for t in TECHS_PRIMARY if t in rate.index]
rate.loc[plot_techs, ['critical', 'flat']].plot(kind='bar', ax=ax, color=['tab:red', 'tab:blue'])
ax.set_title('Mechanical-repeat rate: % of unit-hours where all 4 DA quarter-bids are identical, post-MTU15-DA', fontsize=10)
ax.set_ylabel('% of unit-hours')
ax.set_ylim(0, 100)
ax.tick_params(axis='x', rotation=0)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'bid_atlas_14_mechanical_repeat.png', dpi=110, bbox_inches='tight')
plt.show()


## Synthesis — what the atlas shows

This section is intentionally brief: the atlas is descriptive and the headline interpretation lives in the modelling track and the workshop deck. Key facts to take away:

- **Across regimes**: bid complexity (tranches per offer) and price range (P90) shift through the reform sequence in patterns that are tech- and firm-specific, not uniform. Cross-regime aggregate views hide this heterogeneity.
- **By technology**: CCGT and hydro exhibit the richest bid-shape variation; wind, solar, and nuclear bid simply (1–2 tranches typically) and do not engage with the granularity reform's strategic optionality.
- **Dominant vs Fringe**: dominant firms bid distinct ladders from the fringe across most regimes; the gap widens at MTU15-DA. Cross-validates the fringe placebo (B13).
- **Hour-of-day**: critical hours h{7, 8, 16, 17, 18} show systematically richer ladders than flat hours h{3, 4, 5} where the within-market granularity mechanism predicts they should.
- **Critical-flat differential**: positive and growing at MTU15-DA for CCGT and hydro; ~zero for wind, solar, nuclear. Cross-validates B14 at the descriptive level.
- **Seasonality**: visible patterns in the heatmaps but seasonality does not invert the regime ordering. Same-cal-month robustness in §X.1 of `_modelling_track.md` already established this for the q₂ DiD.
- **Plant-level drilldown**: top CCGT units (TAPOWER, ARCOS1/2/3, BES3/5, etc.) and top hydro units (TAMEGA, SIL, DUER, TAJO) drive most of the aggregate patterns; renewable units bid uniformly.
- **Within-hour granularity exploitation (post-MTU15-DA)**: a substantial fraction of unit-hours in the wind/solar/nuclear category are "mechanical repeats" (all four DA quarter-bids identical), while CCGT and hydro show much lower mechanical-repeat rates in critical hours. Direct visual evidence that operators DO use the within-hour granularity, but only where it has economic content.

These descriptive facts feed into:

- `notebooks/memos/_modelling_track.md` § X (the within-day DiD identification design).
- `notebooks/memos/_within_market_granularity_model.md` (the stylised IO model that should formalise these descriptive patterns).
- `CLAIMS_LEDGER.md` rows B14 (bid-shape rich-ladder pattern) and B12/B13 (the within-day DiD headline + fringe placebo).

**Caveats.** Pre-2025-03-19 DA bid prices are 0-padded due to a parser artefact; the cross-regime view in this atlas uses IDA bids (IDET) for that reason. The within-hour granularity exploitation analysis (§9) uses DA bids restricted to post-MTU15-DA where the four-quarter structure exists.
